# CSV 驱动的全景 SAR 特征提取（v9.1）

## 相对 v8 的变化

1. **按 scene 去重**：先从站点 CSV 汇总全部不重复的 scene_id，每景只计算、只导出一次；不再按“站点 × 景”重复计算同一景。
2. **导出网格中心经纬度**：每行新增 `center_lon`、`center_lat`（EPSG:4326，边界格为 footprint 内部分的质心），用于在本地把网格分配给 MPA 站点。
3. **去掉 site_id**：GEE 端不再携带站点标签；站点归属在本地用中心点与 MPA 面做空间连接得到。
4. **分批、可续跑的提交**：`submit_scenes` 每次最多提交 `MAX_SUBMIT` 景，已提交的写入日志 CSV，重复运行自动跳过，避免超过 GEE 任务队列上限。
5. **导出文件名**改用完整 scene_id，避免截短后重名。
6. **FEATURE_COLUMNS** 需与 Oil_LightGBM 中严格模型的 `selected_feature_names` 保持一致（见参数区注释）。
7. **（v9.1）任务状态追踪与失败重提**：日志记录每次提交的 task_id；`update_scene_status` 查询每景最近一次任务的状态与错误信息，写入状态表；`resubmit_failed` 用更大的 tileScale 重提失败景，重试用尽的写入 `failed_scenes_2015.csv`。所有 `tileScale` 统一由参数 `TILE_SCALE` 控制。
8. 末尾新增**本地后处理示例**（过滤空网格 → 预测 → 分配站点），需在装有 geopandas 的本地环境运行。

分割算法、阈值、Lee 滤波、6 km 网格均值、面积筛选和各项特征的计算方式与 v8 完全一致。

## 尚存资源限制（沿用 v8 说明）

全景几何面积筛选需要整景 10 m 矢量化，可能触发内存/复杂几何/耗时限额。必须在 Tasks 页面观察异步导出的最终状态，不能仅以提交成功认定计算完成。建议先用 `submit_scenes(scenes[:1], ...)` 验证单景导出成功，再分批提交。


In [ ]:
#### CSV 驱动的全景水域 SAR 分割与 6 km 特征提取（v9：按 scene 去重）####
import ee
ee.Authenticate()
ee.Initialize(project='where-has-sar')

import os
os.environ['KMP_DUPLICATE_LIB_OK']='TRUE'

In [6]:
try:
    import fsspec
except ImportError:
    raise ImportError("缺少依赖 fsspec，请在终端执行：\nconda activate oil\npip install fsspec")

In [13]:
import pandas as pd
from pathlib import Path


# ----------------- 核心参数配置：与 Seg_Square 保持一致 -----------------
LEE_KERNEL_RADIUS = 1
LEE_ENL = 4
ABS_THRESHOLD_DB = -21
DB_THRESHOLD_SHIFT = 3
BRIGHT_THRESHOLD = -15
MORPH_RADIUS = 1
MIN_AREA_KM2 = 0.02
MIN_OBJECT_PIXELS = 9
SCENE_VECTOR_MAX_PIXELS = 10_000_000_000  # 全景 10 m 矢量化上限，不改变像元尺度
TILE_SCALE = 8          # 首次提交使用；失败重提时可调大（如 16），由 resubmit_failed 临时修改
FEATURE_SCALE = 10
GRID_SCALE = 6000       # 保留 MPA 单景矩阵降采样框架
GRID_REDUCTION_MAX_PIXELS = 1_000_000  # 每个 6 km 网格的 10 m 统计上限
TEXTURE_SCALE = 10
TEXTURE_LEVELS = 32
BACKGROUND_RING_M = 60
EPS = 1e-6
PI = 3.141592653589793
SAFE_FILL = -999

# 必须与 Oil_LightGBM 中严格模型的 lgbm_artifacts_strict['selected_feature_names'] 一致。
# 未被模型选中的列在推理时可为 NaN（被筛选器丢弃，不影响预测）；
# Year、sar_hour 可在本地由 sar_time 推出，不必依赖 GEE 输出。
# 若选中特征里有本列表之外的特征（如 min_obj），需在 _grid_feature 中补充计算。
FEATURE_COLUMNS = [
    'dark_spot_area_mean_m2', 'vv_filtered_stdDev', 'dark_spot_count',
    'sar_hour', 'corr', 'local_mean_after', 'wind_mean',
    'dark_spot_compactness_area_weighted', 'intensity_ratio',
    'wind_dir_sin', 'asm', 'dark_spot_area_std_m2',
    'dark_spot_lwr_mean', 'isdr', 'isri', 'shade'
]
META_COLUMNS = ['scene_id', 'sar_time', 'center_lon', 'center_lat']

# 唯一用户输入为 CSV（where_has_sar 导出，含分号分隔 scene_ids）；不加载 MPA 矢量资产。
SAR_info_csv = 'C://Users//pppmi//地信//代码//oil//111//data//where_has_sar//sar_2015_protected_areas_with_sceneid.csv'
output_feature = 'MPA_Features_2015'

# 分批提交控制
SUBMIT_LOG = 'submitted_scenes_2015.csv'   # 每次提交一行：scene_id, task_id, 提交时间, tile_scale, 第几次尝试
STATUS_CSV = 'scene_task_status_2015.csv'  # 每景最近一次任务的状态（SUCCEEDED / FAILED / RUNNING ...）
GIVEUP_CSV = 'failed_scenes_2015.csv'      # 重试次数用尽仍失败的景
MAX_SUBMIT = 200                           # 每次运行最多提交的景数

# 自动海陆筛选：MOD44W 保留海洋，JRC 补充已识别地表水。
# JRC occurrence 单独使用不覆盖大部分公海；无数据不能直接视为海洋。
JRC_WATER_OCCURRENCE_MIN = 10
LAND_WATER_REFERENCE_YEAR = 2015


def get_water_mask():
    """全球海陆分类 + JRC 水体；不依赖任何用户上传的面资产。"""
    modis_collection = (ee.ImageCollection('MODIS/006/MOD44W')
                        .filterDate(f'{LAND_WATER_REFERENCE_YEAR}-01-01',
                                    f'{LAND_WATER_REFERENCE_YEAR + 1}-01-01')
                        .select('water_mask')
                        .map(lambda image: ee.Image(image).toByte()))
    # 先放入全陆地兜底影像，再叠加 MOD44W；即使集合为空也不会对 null 调用 select。
    modis_fallback = ee.Image.constant(0).rename('water_mask').toByte()
    modis_water = (ee.ImageCollection([modis_fallback])
                   .merge(modis_collection).mosaic().eq(1))
    occurrence = ee.Image('JRC/GSW1_4/GlobalSurfaceWater').select('occurrence')
    # where 输出使用常数影像的完整二值 mask，缺失 occurrence 不推断为海洋。
    jrc_water = ee.Image.constant(0).where(occurrence.gte(JRC_WATER_OCCURRENCE_MIN), 1)
    return modis_water.Or(jrc_water).rename('water_mask')


def validate_water_mask_source():
    """提交导出前验证海陆掩膜，空集合不再拖到服务端任务才报错。"""
    collection = ee.ImageCollection('MODIS/006/MOD44W').filterDate(
        f'{LAND_WATER_REFERENCE_YEAR}-01-01', f'{LAND_WATER_REFERENCE_YEAR + 1}-01-01'
    )
    count = collection.size().getInfo()
    if count == 0:
        raise ValueError(f'MODIS/006/MOD44W 在 {LAND_WATER_REFERENCE_YEAR} 年没有影像；未提交任务。')
    bands = ee.Image(collection.first()).bandNames().getInfo()
    if 'water_mask' not in bands:
        raise ValueError(f'海陆掩膜缺少 water_mask 波段：{bands}')
    # 实际计算一小块组合掩膜，提前发现 mosaic 的波段类型冲突。
    mask_probe = get_water_mask().reduceRegion(
        reducer=ee.Reducer.first(),
        geometry=ee.Geometry.Point([0, 0]).buffer(500),
        scale=250, maxPixels=256
    ).getInfo()
    if mask_probe.get('water_mask') is None:
        raise ValueError(f'组合水体掩膜检查返回空值：{mask_probe}')
    return {'collection': 'MODIS/006/MOD44W', 'year': LAND_WATER_REFERENCE_YEAR, 'count': count, 'bands': bands, 'mask_probe': mask_probe}


def _safe_num(x, default=SAFE_FILL):
    return ee.Number(ee.List([x, default]).reduce(ee.Reducer.firstNonNull()))


def _safe_div(a, b, default=SAFE_FILL):
    a = ee.Number(a)
    b = ee.Number(b)
    return ee.Number(ee.Algorithms.If(b.abs().gt(EPS), a.divide(b), default))


def lee_filter(img_db):
    """与 Seg_Square 一致：功率域局部统计 MMSE Lee 滤波，返回 dB。"""
    img_db = img_db.toFloat()
    # exp(dB × ln(10)/10) = 10**(dB/10)：同一功率转换公式。
    # 从 SAR 影像本身开始，保留原生投影、footprint 和 mask，不继承常数影像的全球范围。
    img_power = img_db.divide(10).multiply(2.302585092994046).exp()
    kernel = ee.Kernel.square(radius=LEE_KERNEL_RADIUS, units='pixels')
    mean_power = img_power.reduceNeighborhood(ee.Reducer.mean(), kernel)
    mean_sq = img_power.pow(2).reduceNeighborhood(ee.Reducer.mean(), kernel)
    var_power = mean_sq.subtract(mean_power.pow(2)).max(0)
    noise_var = mean_power.pow(2).divide(LEE_ENL)
    signal_var = var_power.subtract(noise_var).max(0)
    weights = signal_var.divide(var_power.max(1e-12)).clamp(0, 1)
    filtered_power = mean_power.add(weights.multiply(img_power.subtract(mean_power))).max(1e-12)
    return filtered_power.log10().multiply(10).rename('VV')


def get_era5_wind_image(sar_time, region):
    """匹配 SAR 成像小时的 ERA5 风场，返回风速及风向 sin 分量。"""
    hour_start = ee.Date.fromYMD(
        sar_time.get('year'), sar_time.get('month'), sar_time.get('day')
    ).advance(sar_time.get('hour'), 'hour')
    hour_end = hour_start.advance(1, 'hour')
    era5_collection = (ee.ImageCollection('ECMWF/ERA5/HOURLY')
                       .filterBounds(region)
                       .filterDate(hour_start, hour_end)
                       .select(['u_component_of_wind_10m', 'v_component_of_wind_10m']))
    # 有数据时必须取真实 ERA5，不能把兜底放首位后直接 first()。
    # 缺失时使用全掩膜波段，输出缺失值，而不是伪造零风速。
    era5_fallback = (ee.Image.constant([0, 0])
                     .rename(['u_component_of_wind_10m', 'v_component_of_wind_10m'])
                     .updateMask(ee.Image.constant(0)))
    era5 = ee.Image(ee.Algorithms.If(
        era5_collection.size().gt(0), era5_collection.first(), era5_fallback
    ))
    u_img = era5.select('u_component_of_wind_10m')
    v_img = era5.select('v_component_of_wind_10m')
    speed_img = u_img.pow(2).add(v_img.pow(2)).sqrt().rename('wind_speed')
    wind_dir_sin_img = v_img.divide(speed_img.max(1e-6)).rename('wind_dir_sin')
    return speed_img.addBands(wind_dir_sin_img)


def _coarse_water_mean(img, band_name, region=None, grid=None, return_stats=False):
    """按原景 6 km 网格对 10 m 水域像元求均值，避免强制生成全景细栅格。"""
    native_proj = img.projection()
    coarse_proj = native_proj.atScale(GRID_SCALE)
    region = img.geometry() if region is None else region
    grid = region.coveringGrid(coarse_proj) if grid is None else grid
    # 单格 600×600=360000 像元，上限留边界余量；没有 bestEffort 降分辨率。
    # 单输入 reduceRegions 的字段显式命名为 VV，避免读取错误字段。
    stats = img.reduceRegions(
        collection=grid, reducer=ee.Reducer.mean().setOutputs(['VV']),
        crs=native_proj, scale=FEATURE_SCALE, tileScale=TILE_SCALE,
        maxPixelsPerRegion=GRID_REDUCTION_MAX_PIXELS
    ).filter(ee.Filter.notNull(['VV']))
    # 矩形直接 reduceToImage 会在公共边界混入邻格属性；只在格中心放均值。
    centers = stats.map(lambda f: ee.Feature(
        ee.Feature(f).geometry().centroid(1, coarse_proj), {'VV': ee.Feature(f).get('VV')}
    ))
    # 此处唯一强制重投影的对象是稀疏格中心属性，输出为粗网格（不是 10 m SAR）。
    coarse_image = centers.reduceToImage(['VV'], ee.Reducer.first()).rename(band_name).reproject(coarse_proj)
    if return_stats:
        grid_stats = stats.map(lambda f: ee.Feature(f).set(band_name, ee.Feature(f).get('VV')))
        return coarse_image, grid_stats
    return coarse_image


def matrix_pooling_segmentation(sar_filtered, region, grid=None, return_stats=False):
    """景内 6 km 矩阵降采样两步阈值分割；阈值及形态学与 Seg_Square 一致。"""
    mean1_coarse = _coarse_water_mean(sar_filtered, 'local_mean_before', region, grid)
    mean1_10m = mean1_coarse.resample('bilinear')

    # 与 Square 一致：亮像元定义为 > -15 dB，替换为第一次局部均值。
    bright_mask = sar_filtered.gt(BRIGHT_THRESHOLD)
    img_suppressed = sar_filtered.where(bright_mask, mean1_10m).rename('VV')
    mean2_coarse, mean2_stats = _coarse_water_mean(
        img_suppressed, 'local_mean_after', region, grid, return_stats=True
    )
    mean2_10m = mean2_coarse.resample('bilinear')

    condition1 = img_suppressed.lt(mean2_10m.subtract(DB_THRESHOLD_SHIFT))
    condition2 = img_suppressed.lt(ABS_THRESHOLD_DB)
    oil_mask_raw = condition1.And(condition2).rename('oil_mask')
    kernel = ee.Kernel.square(MORPH_RADIUS)
    oil_mask_morph = oil_mask_raw.focal_max(kernel=kernel).focal_min(kernel=kernel)
    # 形态学不能把油膜扩回陆地或 SAR 无效像元。
    oil_mask_morph = oil_mask_morph.updateMask(sar_filtered.mask())

    # 与 Square 一致：矢量化连通斑块并剔除面积小于 0.02 km² 的斑块。
    patch_vectors = oil_mask_morph.selfMask().reduceToVectors(
        geometry=region, crs=sar_filtered.projection(), scale=FEATURE_SCALE, geometryType='polygon',
        eightConnected=True, labelProperty='label',
        maxPixels=SCENE_VECTOR_MAX_PIXELS, bestEffort=False, tileScale=TILE_SCALE
    )
    patch_vectors = patch_vectors.map(
        lambda f: ee.Feature(f).set(
            'patch_area_km2', ee.Feature(f).geometry().area(1).divide(1e6)
        )
    )
    big_patches = patch_vectors.filter(ee.Filter.gte('patch_area_km2', MIN_AREA_KM2))
    oil_mask_final = (sar_filtered.multiply(0).unmask(0).byte()
                      .paint(big_patches, 1)
                      .And(oil_mask_morph).unmask(0).rename('oil_mask')
                      .setDefaultProjection(sar_filtered.projection()).clip(region))
    if return_stats:
        return oil_mask_final, img_suppressed, mean2_coarse, mean2_stats
    return oil_mask_final, img_suppressed, mean2_coarse


def _multi_dark_spot_features(patches):
    """复用当前网格已经矢量化的斑块，汇总面积、数量、紧凑度和长宽比。"""
    def add_patch_geometry(feature):
        geom = ee.Feature(feature).geometry()
        area_m2 = geom.area(1)
        perimeter_m = geom.perimeter(1)
        centroid = geom.centroid(1).coordinates()
        latitude = ee.Number(centroid.get(1))
        bounds = ee.List(geom.bounds(1).coordinates().get(0))
        p0, p1, p2 = ee.List(bounds.get(0)), ee.List(bounds.get(1)), ee.List(bounds.get(2))
        # 经度方向距离按中心纬度修正，不使用与纬度无关的固定经度换算。
        width_m = (ee.Number(p1.get(0)).subtract(ee.Number(p0.get(0))).abs()
                   .multiply(111320).multiply(latitude.multiply(PI / 180).cos().abs()))
        height_m = ee.Number(p2.get(1)).subtract(ee.Number(p1.get(1))).abs().multiply(111320)
        lwr = _safe_div(width_m.max(height_m), width_m.min(height_m), 0)
        compactness = _safe_div(
            ee.Number(4).multiply(PI).multiply(area_m2), perimeter_m.pow(2), 0
        )
        return ee.Feature(feature).set({
            'patch_area_m2': area_m2,
            'patch_lwr': lwr,
            'patch_compactness': compactness
        })

    patches = patches.map(add_patch_geometry)
    area_stats = ee.Dictionary(patches.aggregate_stats('patch_area_m2'))
    lwr_stats = ee.Dictionary(patches.aggregate_stats('patch_lwr'))
    total_area = _safe_num(patches.aggregate_sum('patch_area_m2'), 0)
    weighted = patches.map(lambda f: ee.Feature(f).set(
        'compactness_x_area',
        ee.Number(ee.Feature(f).get('patch_compactness'))
        .multiply(ee.Number(ee.Feature(f).get('patch_area_m2')))
    ))
    return ee.Dictionary({
        'dark_spot_area_mean_m2': _safe_num(area_stats.get('mean'), 0),
        'dark_spot_count': patches.size(),
        'dark_spot_compactness_area_weighted': _safe_div(
            _safe_num(weighted.aggregate_sum('compactness_x_area'), 0), total_area, 0
        ),
        'dark_spot_area_std_m2': _safe_num(area_stats.get('sample_sd'), 0),
        'dark_spot_lwr_mean': _safe_num(lwr_stats.get('mean'), 0)
    })


def _grid_feature(grid, oil_mask, raw_img, filtered_img, mean2_coarse,
                  wind_img, texture_img, sar_time, scene_id):
    """原生 6 km 网格；空暗斑分支不触发矢量化、缓冲和对象纹理统计。
    v9：不再携带 site_id，改为输出网格中心经纬度，供本地分配站点。"""
    grid = ee.Feature(grid)
    region = grid.geometry()
    sar_proj = raw_img.projection()
    pixel_count = _safe_num(oil_mask.eq(1).reduceRegion(
        reducer=ee.Reducer.sum(), geometry=region, crs=sar_proj, scale=FEATURE_SCALE,
        maxPixels=1e8, tileScale=TILE_SCALE
    ).get('oil_mask'), 0)
    vv_stats = filtered_img.reduceRegion(
        reducer=ee.Reducer.mean().combine(ee.Reducer.stdDev(), '', True),
        geometry=region, crs=sar_proj, scale=FEATURE_SCALE,
        maxPixels=1e8, tileScale=TILE_SCALE
    )
    wind_stats = wind_img.reduceRegion(
        reducer=ee.Reducer.mean(), geometry=region, scale=1000, maxPixels=1e6
    )
    # 直接读取本格第二次均值，避免边界格中心不在 footprint 时返回空值。

    window_dict = ee.Dictionary({
        'vv_filtered_stdDev': _safe_num(vv_stats.get('VV_stdDev')),
        'sar_hour': sar_time.get('hour'),
        'local_mean_after': _safe_num(grid.get('local_mean_after')),
        'wind_mean': _safe_num(wind_stats.get('wind_speed')),
        'wind_dir_sin': _safe_num(wind_stats.get('wind_dir_sin'), 0)
    })

    def when_valid_object():
        obj = oil_mask.eq(1).selfMask().clip(region)
        patches = oil_mask.selfMask().reduceToVectors(
            geometry=region, crs=sar_proj, scale=FEATURE_SCALE,
            geometryType='polygon', eightConnected=True, labelProperty='label',
            bestEffort=False, maxPixels=1e8, tileScale=TILE_SCALE
        )
        # 本格只矢量化一次，形状统计与 60 m 几何背景环复用它。
        dark_spot_dict = _multi_dark_spot_features(patches)
        obj_geom = patches.geometry(1)
        outer_ring = obj_geom.buffer(BACKGROUND_RING_M).difference(obj_geom, 1)
        background_mask = oil_mask.eq(1).unmask(0).Not()
        obj_stats = raw_img.updateMask(obj).reduceRegion(
            reducer=ee.Reducer.mean().combine(ee.Reducer.stdDev(), '', True),
            geometry=region, crs=sar_proj, scale=FEATURE_SCALE,
            maxPixels=1e8, tileScale=TILE_SCALE
        )
        bg_stats = raw_img.updateMask(background_mask).clip(region).reduceRegion(
            reducer=ee.Reducer.mean().combine(ee.Reducer.stdDev(), '', True),
            geometry=outer_ring, crs=sar_proj, scale=FEATURE_SCALE,
            maxPixels=1e8, tileScale=TILE_SCALE
        )
        fallback_bg_stats = raw_img.updateMask(background_mask).reduceRegion(
            reducer=ee.Reducer.mean().combine(ee.Reducer.stdDev(), '', True),
            geometry=region, crs=sar_proj, scale=FEATURE_SCALE,
            maxPixels=1e8, tileScale=TILE_SCALE
        )
        mu_sce = _safe_num(ee.List([
            bg_stats.get('VV_mean'), fallback_bg_stats.get('VV_mean')
        ]).reduce(ee.Reducer.firstNonNull()))
        sigma_sce = _safe_num(ee.List([
            bg_stats.get('VV_stdDev'), fallback_bg_stats.get('VV_stdDev')
        ]).reduce(ee.Reducer.firstNonNull()))
        mu_obj = _safe_num(obj_stats.get('VV_mean'), 0)
        sigma_obj = _safe_num(obj_stats.get('VV_stdDev'), 0)
        tex_stats = texture_img.updateMask(obj).reduceRegion(
            reducer=ee.Reducer.mean(), geometry=region, crs=sar_proj, scale=TEXTURE_SCALE,
            maxPixels=1e8, tileScale=TILE_SCALE
        )
        return dark_spot_dict.combine(ee.Dictionary({
            'corr': _safe_num(tex_stats.get('VV_corr')),
            'asm': _safe_num(tex_stats.get('VV_asm')),
            'shade': _safe_num(tex_stats.get('VV_shade')),
            'intensity_ratio': _safe_div(mu_obj, mu_sce, 0),
            'isdr': _safe_div(sigma_obj, sigma_sce, 0),
            'isri': _safe_div(mu_obj, sigma_obj, 0)
        }))

    empty_defaults = ee.Dictionary({
        'dark_spot_area_mean_m2': 0, 'dark_spot_count': 0,
        'dark_spot_compactness_area_weighted': 0, 'dark_spot_area_std_m2': 0,
        'dark_spot_lwr_mean': 0, 'corr': SAFE_FILL, 'asm': SAFE_FILL,
        'shade': SAFE_FILL, 'intensity_ratio': 0, 'isdr': 0, 'isri': 0
    })
    object_dict = ee.Dictionary(ee.Algorithms.If(
        pixel_count.gte(MIN_OBJECT_PIXELS), when_valid_object(), empty_defaults
    ))
    # v9 新增：网格中心经纬度（EPSG:4326）；边界格为 footprint 内部分的质心。
    center = region.centroid(1).coordinates()
    # 只导出属性，不导出几何，减小 CSV 体积。
    return ee.Feature(None, window_dict.combine(object_dict)).set({
        'scene_id': scene_id,
        'sar_time': sar_time.format('YYYY-MM-dd HH:mm:ss'),
        'center_lon': center.get(0),
        'center_lat': center.get(1)
    })


def process_single_scene(scene_id):
    """CSV 驱动全景处理（v9：每景只算一次，不携带 site_id）。"""
    scene_id = str(scene_id).strip()
    image_id = (scene_id if scene_id.startswith('COPERNICUS/S1_GRD/')
                else f'COPERNICUS/S1_GRD/{scene_id}')
    s1_image = ee.Image(image_id)
    sar_time = ee.Date(s1_image.get('system:time_start'))
    scene_geom = s1_image.geometry()
    sar_db = s1_image.select('VV').rename('VV')
    # 先剔除陆地，再进行 Lee 统计、两次粗网格均值及所有后续特征统计。
    raw_img = sar_db.updateMask(get_water_mask()).clip(scene_geom)
    # 显式保留原景 VV 投影，避免常数影像参与运算后落到默认投影。
    sar_filtered = lee_filter(raw_img).setDefaultProjection(sar_db.projection())

    coarse_proj = sar_db.projection().atScale(GRID_SCALE)
    # 完整方格用于均值统计；SAR 没有被分块裁剪，跨格滤波与形态学仍连续。
    native_grid = scene_geom.coveringGrid(coarse_proj)
    oil_mask, img_suppressed, mean2_coarse, valid_grid = matrix_pooling_segmentation(
        sar_filtered, scene_geom, native_grid, return_stats=True
    )
    wind_img = get_era5_wind_image(sar_time, scene_geom)
    # 单景只定义一份 GLCM；不先裁剪网格，保持纹理邻域连续。
    texture_base = (raw_img.clamp(-35, 0).unitScale(-35, 0)
                    .multiply(TEXTURE_LEVELS - 1).toInt().rename('VV'))
    texture_img = texture_base.glcmTexture(size=1).select(['VV_corr', 'VV_asm', 'VV_shade'])

    # 复用第二次均值统计的网格属性，不再对均值图逐格 reduceRegion。
    # 边界格统计限定到 footprint；整景图像及其跨格邻域不做独立裁剪。
    grid_polys = valid_grid.map(lambda f: ee.Feature(
        ee.Feature(f).geometry().intersection(scene_geom, 1),
        {'local_mean_after': ee.Feature(f).get('local_mean_after')}
    ))
    return grid_polys.map(lambda grid: _grid_feature(
        grid, oil_mask, raw_img, sar_filtered, mean2_coarse,
        wind_img, texture_img, sar_time, scene_id
    ))


In [15]:
# ----------------- v9.1：按 scene 去重 + 分批可续跑提交 + 任务状态追踪 -----------------
import csv
from datetime import datetime

LOG_FIELDS = ['scene_id', 'task_id', 'submit_time', 'tile_scale', 'attempt']


def collect_unique_scenes(csv_path):
    """从站点 CSV 汇总全部不重复的 scene_id（同一景覆盖多个站点时只保留一次）。"""
    df = pd.read_csv(csv_path, encoding='utf-8-sig')
    if 'scene_ids' not in df.columns:
        raise ValueError('CSV 缺少字段：scene_ids')
    df = df.dropna(subset=['scene_ids'])
    if 'has_image' in df.columns:
        has_image = df['has_image'].astype(str).str.strip().str.lower()
        df = df[has_image.isin(['1', 'true'])]
    scenes = df['scene_ids'].astype(str).str.split(';').explode().str.strip()
    scenes = scenes[scenes.ne('') & scenes.ne('nan')]
    return sorted(set(scenes))


def _read_submit_log(log_path=SUBMIT_LOG):
    log = Path(log_path)
    if not log.exists():
        return pd.DataFrame(columns=LOG_FIELDS)
    return pd.read_csv(log, dtype=str)


def _append_submit_log(row, log_path=SUBMIT_LOG):
    log = Path(log_path)
    is_new = not log.exists()
    with log.open('a', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=LOG_FIELDS)
        if is_new:
            writer.writeheader()
        writer.writerow(row)


def _start_scene_task(scene_id, folder, attempt, log_path=SUBMIT_LOG):
    """提交单景导出任务，并把 task_id 记入日志，之后按 task_id 精确查询状态。"""
    task = ee.batch.Export.table.toDrive(
        collection=process_single_scene(scene_id),
        description=scene_id[:100],        # GEE 任务名上限 100 字符
        folder=folder,
        fileNamePrefix=scene_id,           # 完整 scene_id，避免重名
        fileFormat='CSV',
        selectors=META_COLUMNS + FEATURE_COLUMNS
    )
    task.start()
    _append_submit_log({
        'scene_id': scene_id, 'task_id': task.id,
        'submit_time': datetime.now().isoformat(timespec='seconds'),
        'tile_scale': TILE_SCALE, 'attempt': attempt
    }, log_path)
    return task.id


def submit_scenes(scenes, folder, log_path=SUBMIT_LOG, max_submit=MAX_SUBMIT):
    """首次提交：每次最多 max_submit 景；日志中已有的景自动跳过（失败重提请用 resubmit_failed）。"""
    validate_water_mask_source()
    done = set(_read_submit_log(log_path)['scene_id'])
    pending = [s for s in scenes if s not in done]
    todo = pending[:max_submit]

    n_ok = 0
    for i, scene_id in enumerate(todo, 1):
        try:
            _start_scene_task(scene_id, folder, attempt=1, log_path=log_path)
            n_ok += 1
            print(f'[{i}/{len(todo)}] 已提交: {scene_id}')
        except Exception as e:
            # 提交阶段失败（如参数/网络错误）不写日志，下次运行会自动重试
            print(f'[{i}/{len(todo)}] 提交失败: {scene_id} -> {e}')
    print(f'本次成功提交 {n_ok} 景，剩余 {len(pending) - n_ok} 景未提交。')


def fetch_task_states():
    """读取当前账号下 GEE 任务的状态与错误信息。"""
    rows = []
    for op in ee.data.listOperations():
        meta = op.get('metadata', {})
        rows.append({
            'task_id': op.get('name', '').split('/')[-1],
            'state': meta.get('state'),
            'error_message': (op.get('error') or {}).get('message', ''),
            'update_time': meta.get('updateTime', '')
        })
    return pd.DataFrame(rows, columns=['task_id', 'state', 'error_message', 'update_time'])


def update_scene_status(log_path=SUBMIT_LOG, status_csv=STATUS_CSV):
    """汇总每景最近一次任务的状态，写入 status_csv。
    GEE 任务历史不会永久保留，查不到时沿用上次记录的状态；建议定期运行。"""
    log = _read_submit_log(log_path)
    if log.empty:
        print('尚无提交记录。')
        return None
    latest = log.groupby('scene_id', sort=False).tail(1)   # 日志按提交顺序追加，取每景最后一行即最近一次
    merged = latest.merge(fetch_task_states(), on='task_id', how='left')

    status_path = Path(status_csv)
    if status_path.exists():
        old = pd.read_csv(status_path, dtype=str).drop_duplicates('task_id').set_index('task_id')
        lost = merged['state'].isna()
        for col in ['state', 'error_message', 'update_time']:
            if col in old.columns:
                merged.loc[lost, col] = merged.loc[lost, 'task_id'].map(old[col])
    merged['state'] = merged['state'].fillna('NOT_FOUND')
    merged['error_message'] = merged['error_message'].fillna('')
    merged.to_csv(status_csv, index=False, encoding='utf-8-sig')

    print('各状态景数：')
    print(merged['state'].value_counts().to_string())
    failed = merged[merged['state'] == 'FAILED']
    if len(failed):
        print('\n失败原因（前 10 类）：')
        print(failed['error_message'].str[:80].value_counts().head(10).to_string())
    return merged


def resubmit_failed(folder, status_csv=STATUS_CSV, tile_scale=16, max_attempts=2,
                    retry_states=('FAILED',), max_submit=MAX_SUBMIT, giveup_csv=GIVEUP_CSV):
    """把失败的景用更大的 tileScale 重新提交；达到 max_attempts 仍失败的写入 giveup_csv。
    注意：tileScale 只能缓解部分内存问题，整景矢量化仍可能失败，需人工决定后续处理。"""
    global TILE_SCALE
    status = pd.read_csv(status_csv, dtype=str)
    status['attempt'] = status['attempt'].astype(int)
    bad = status[status['state'].isin(retry_states)]
    retry = bad[bad['attempt'] < max_attempts].head(max_submit)
    giveup = bad[bad['attempt'] >= max_attempts]

    old_tile_scale = TILE_SCALE
    TILE_SCALE = tile_scale
    n_ok = 0
    try:
        for scene_id, attempt in retry[['scene_id', 'attempt']].itertuples(index=False):
            try:
                _start_scene_task(scene_id, folder, attempt=attempt + 1)
                n_ok += 1
                print(f'已重新提交（第 {attempt + 1} 次，tileScale={tile_scale}）: {scene_id}')
            except Exception as e:
                print(f'重新提交失败: {scene_id} -> {e}')
    finally:
        TILE_SCALE = old_tile_scale

    giveup[['scene_id', 'task_id', 'attempt', 'tile_scale', 'error_message']].to_csv(
        giveup_csv, index=False, encoding='utf-8-sig')
    print(f'本次重新提交 {n_ok} 景；重试已用尽的 {len(giveup)} 景已写入 {giveup_csv}')


In [18]:
scenes = collect_unique_scenes(SAR_info_csv)
print('去重后影像数：', len(scenes))

# 第一步：先只提交 1 景，到 Tasks 页面确认 COMPLETED 且 CSV 内容正确。
# submit_scenes(scenes[:1], output_feature)

# 第二步：确认无误后分批提交；等队列消化后再次运行本格即可续跑。
submit_scenes(scenes, output_feature)


去重后影像数： 19322
[1/200] 已提交: S1A_EW_GRDM_1SDV_20150109T181326_20150109T181421_004098_004F5A_E165
[2/200] 已提交: S1A_EW_GRDM_1SDV_20150110T060544_20150110T060617_004105_004F88_358F
[3/200] 已提交: S1A_EW_GRDM_1SDV_20150110T060636_20150110T060703_004105_004F89_A276
[4/200] 已提交: S1A_EW_GRDM_1SDV_20150110T060859_20150110T061003_004105_004F8B_A423
[5/200] 已提交: S1A_EW_GRDM_1SDV_20150110T061003_20150110T061103_004105_004F8B_9558
[6/200] 已提交: S1A_EW_GRDM_1SDV_20150110T165114_20150110T165219_004111_004FAD_294F
[7/200] 已提交: S1A_EW_GRDM_1SDV_20150110T165219_20150110T165319_004111_004FAD_442A
[8/200] 已提交: S1A_EW_GRDM_1SDV_20150111T064547_20150111T064647_004120_004FDA_395D
[9/200] 已提交: S1A_EW_GRDM_1SDV_20150111T064747_20150111T064809_004120_004FDA_777F
[10/200] 已提交: S1A_EW_GRDM_1SDV_20150111T175257_20150111T175401_004127_004FFD_1451
[11/200] 已提交: S1A_EW_GRDM_1SDV_20150111T175402_20150111T175453_004127_004FFD_3220
[12/200] 已提交: S1A_EW_GRDM_1SDV_20150111T175728_20150111T175801_004127_005001_6FC6
[13/200] 已提

In [19]:
# 第三步：定期查看任务状态（建议每天跑一次，GEE 任务历史不会永久保留）
status = update_scene_status()

# 第四步：把失败的景用更大的 tileScale 重新提交（默认最多尝试 2 次）
#resubmit_failed(output_feature, tile_scale=16, max_attempts=2)
# 重提后过一段时间再运行 update_scene_status() 查看结果


各状态景数：
state
PENDING      194
RUNNING        4
SUCCEEDED      2
FAILED         1

失败原因（前 10 类）：
error_message
Image.reduceToVectors: Too many pixels in the region. Found 1728340488, but maxP    1


## 本地后处理示例（不在 GEE 中运行）

以下代码需在装有 geopandas、且已训练好严格模型的本地环境运行（可复制到 Oil_LightGBM 末尾）：

0. 核对：每景成功导出都会生成一个 CSV（没有暗斑的网格也会输出一行），所以“状态表里成功、却没有文件”或“没有文件也没有成功记录”的景都需要排查；
1. 合并全部景的 CSV；
2. 过滤空网格（`dark_spot_count > 0`，与训练集 `ok_34f` 口径一致）；
3. 每个网格只预测一次；
4. 用网格中心点与 MPA 面做空间连接，分配站点。

MPA 面图层需先从 GEE 资产 `Navigator_AllSites_2604_final_keep` 导出一次（SHP / GeoJSON）。
注意：按“中心点落在 MPA 内”判断时，小于 6 km 的保护区可能没有网格中心落入；如需要，可改为“中心点 3 km 缓冲与 MPA 相交”。

In [ ]:
# ===== 本地运行：合并 → 过滤空网格 → 预测 → 分配站点 =====
import glob
import numpy as np
import pandas as pd
import geopandas as gpd
from pathlib import Path

FEATURE_DIR = r'F:/MYR/Research_/ZHANG/Oiling/results/MPA_Features_2015'   # Drive 下载后的目录
MPA_PATH = r'F:/MYR/Research_/ZHANG/Oiling/data/Navigator_AllSites.shp'     # 从 GEE 导出的 MPA 面
OUT_PATH = r'F:/MYR/Research_/ZHANG/Oiling/results/MPA_grid_predictions_2015.csv'

STATUS_PATH = r'F:/MYR/Research_/ZHANG/Oiling/results/scene_task_status_2015.csv'  # GEE 端生成的状态表

# 0) 核对：状态表 vs Drive 实际文件
csv_files = glob.glob(f'{FEATURE_DIR}/*.csv')
have_file = {Path(p).stem for p in csv_files}
status = pd.read_csv(STATUS_PATH, dtype=str)
status['has_file'] = status['scene_id'].isin(have_file)
print(pd.crosstab(status['state'], status['has_file']))
status[(status['state'] != 'SUCCEEDED') | (~status['has_file'])].to_csv(
    STATUS_PATH.replace('.csv', '_need_check.csv'), index=False, encoding='utf-8-sig')

grid = pd.concat((pd.read_csv(p) for p in csv_files), ignore_index=True)
print('全部网格行数：', len(grid))

# 1) 过滤空网格：只保留有有效暗斑的网格
grid = grid[grid['dark_spot_count'] > 0].copy()
print('含有效暗斑的网格行数：', len(grid))

# 2) 预测（需在 Oil_LightGBM 中已得到 lgbm_artifacts_strict 与 add_time_features）
grid = add_time_features(grid)
grid['Year'] = pd.to_datetime(grid['sar_time']).dt.year   # 若模型选中 Year
artifacts = lgbm_artifacts_strict
for c in artifacts['kept_feature_columns']:
    if c not in grid.columns:
        grid[c] = np.nan   # 未被选中的列为 NaN 不影响预测；被选中的列必须真实存在
missing_selected = [c for c in artifacts['selected_feature_names'] if c not in grid.columns or grid[c].isna().all()]
if missing_selected:
    raise ValueError(f'模型选中的特征在 GEE 输出中缺失：{missing_selected}')
grid['prob_positive'] = artifacts['pipeline'].predict_proba(grid[artifacts['kept_feature_columns']])[:, 1]
grid['pred_label'] = (grid['prob_positive'] >= 0.5).astype(int)

# 3) 分配站点：网格中心点落在 MPA 面内
pts = gpd.GeoDataFrame(grid, crs=4326,
                       geometry=gpd.points_from_xy(grid['center_lon'], grid['center_lat']))
mpa = gpd.read_file(MPA_PATH)[['SITE_ID', 'geometry']].to_crs(4326)
grid_site = gpd.sjoin(pts, mpa, predicate='within', how='inner')   # 重叠 MPA 会对应多行

out_cols = ['SITE_ID', 'scene_id', 'sar_time', 'center_lon', 'center_lat', 'prob_positive', 'pred_label']
grid_site[out_cols].to_csv(OUT_PATH, index=False, encoding='utf-8-sig')
print('已分配到站点的网格行数：', len(grid_site), '->', OUT_PATH)
